# InferRef3D v1.0 — custom model inference

Apply one trusted `trainref3d-model-1.0` binary model to one SegRef3D inference request. Select **Runtime → Change runtime type → T4 GPU**. No automatic uploads occur.

The Model ZIP and Request ZIP are uploaded only when you run the explicit upload cell. Image data may remain identifiable; confirm that your institution permits Google Colab use. Predictions require human review and correction and are not independent clinical diagnoses.

In [ ]:
# Pin this to a reviewed commit SHA for a reproducible run.
BACKEND_REF = 'main'


## 1. Set up the T4 runtime

Downloads code only. Backend hashes and runtime versions are recorded in the result manifest.

In [ ]:
%pip -q install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 monai==1.5.1 nibabel==5.3.2 scipy==1.18.1
from pathlib import Path
from urllib.request import urlopen
import hashlib, importlib, torch
assert torch.__version__.split('+')[0] == '2.8.0', 'Restart the runtime after installation, then run all cells again.'
assert torch.cuda.is_available(), 'Select a T4 GPU: Runtime > Change runtime type.'
for name in ('trainref3d_backend.py', 'inferref3d_backend.py'):
    path = Path('/content') / name
    if BACKEND_REF == 'local':
        assert path.is_file(), f'Upload reviewed {name} via the Colab Files sidebar first.'
        source = path.read_bytes()
    else:
        url = f'https://raw.githubusercontent.com/SatoruMuro/SegRef3D/{BACKEND_REF}/ColabNotebooks/{name}'
        with urlopen(url, timeout=60) as response: source = response.read()
        path.write_bytes(source)
    print(name, 'SHA256:', hashlib.sha256(source).hexdigest())
import trainref3d_backend as tr
import inferref3d_backend as ir
tr, ir = importlib.reload(tr), importlib.reload(ir)
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)


## 2. Explicitly upload Model ZIP + Inference Request ZIP

Create the request in SegRef3D Lite. Upload both ZIPs to your own runtime; nothing is sent to a SegRef3D server.

In [ ]:
from google.colab import files
assert input('I am authorized to upload these files to Google Colab (type YES): ').strip() == 'YES', 'Upload cancelled.'
uploaded = files.upload()
assert len(uploaded) == 2, 'Upload exactly one TrainRef3D Model ZIP and one Inference Request ZIP.'
upload_paths = []
for index, (name, data) in enumerate(uploaded.items()):
    assert name.lower().endswith('.zip'), f'Expected ZIP: {name}'
    path = Path(f'/content/inferref3d_upload_{index}.zip')
    path.write_bytes(data); upload_paths.append(path)
del uploaded, data


## 3. Identify and validate both archives

The model hash, embedded model manifest, target, channels, source category, NIfTI hashes, and original geometry must agree.

In [ ]:
import zipfile
model_path = request_path = None
for path in upload_paths:
    with zipfile.ZipFile(path) as archive: names = set(archive.namelist())
    if 'model_manifest.json' in names: model_path = path
    if 'request_manifest.json' in names: request_path = path
assert model_path and request_path, 'Could not identify one Model ZIP and one Inference Request ZIP.'
model_info = ir.load_model_zip(model_path, '/content')
request_info = ir.load_request_zip(request_path, '/content')
ir.validate_model_request(model_info, request_info)
print('Model:', model_info['manifest']['model_id'])
print('Target:', model_info['manifest']['task'])
print('Source:', request_info['manifest']['input']['source_category'], request_info['manifest']['geometry'])


## 4. GPU inference and exact original-grid restoration

Uses the model manifest preprocessing and sliding-window settings. The class-1 prediction is mapped to the original target Obj ID and nearest-neighbor resampled to the original shape and affine.

In [ ]:
result = ir.run_inference(model_path, request_path, '/content/inferref3d_output')
print('Result ZIP:', result['result_zip'])
print('Peak GPU memory bytes:', result['peak_gpu_memory_bytes'])


## 5. Download and import for review

Download the Result ZIP, then use **Custom Model → Import Prediction ZIP** in SegRef3D Lite. Other object labels are protected during Merge/Replace.

In [ ]:
files.download(result['result_zip'])
